# Automotive Data Mapper - MVP: Exploratory Data Analysis (EDA)

Date: 2026-08-06  
Author: Luis Renteria Lezano  
[LinkedIn](https://www.linkedin.com/in/renteria-luis) | [GitHub](https://github.com/renteria-luis) | [Portfolio](https://luisrenteria.me)

## Executive Summary

- Goal: explore the raw automotive service-record feeds before mapping them into one canonical schema, starting with the data types and formatting problems found in the Shop A feed. This notebook is the first step of the exploratory phase of a vehicle history record (VHR) style data-mapping project.
- **Sources:** Three synthetic vehicle service feeds representing an **independent repair shop**, a **dealership management system**, and a **fleet maintenance provider**:
  - `shop_a`: CSV
  - `dealer_b`: XML
  - `fleet_c`: JSON
- **Data:** [`../data/raw/`](https://github.com/renteria-luis/automotive-data-mapper/tree/main/data/raw)
- **Data dictionary:** [`../docs/DATA_DICTIONARY.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/DATA_DICTIONARY.md)
- **Sample data documentation:** [`../docs/SAMPLE_DATA.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/SAMPLE_DATA.md)

## 0. Reproducibility & Environment Setup

Imports, routes, versions.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import csv

import pandas as pd
from src.profiling import profile


ROOT = Path('..')

RAW = ROOT / "data" / "raw"
SHOP_A = RAW / "shop_a" / "service_records_20260731.csv"
DEALER_B = RAW / "dealer_b" / "ProcessRepairOrder_20260731.xml"
FLEET_C = RAW / "fleet_c" / "maintenance_events_2026-07.json"

for p in (SHOP_A, DEALER_B, FLEET_C):
    print(p.exists(), p)

True ../data/raw/shop_a/service_records_20260731.csv
True ../data/raw/dealer_b/ProcessRepairOrder_20260731.xml
True ../data/raw/fleet_c/maintenance_events_2026-07.json


## 1. An independent auto repair shop: `shop_a` - CSV
42 rows, 24 columns. 

### 1.1 Raw look
Before reading it with pandas, inspect the file as plain text. This is not just ceremony: it confirms four things that `read_csv` takes for granted and that, if they are wrong, can cause it to fail **without warning**.

* What the separator is
* How values containing that separator are marked
* How lines end
* Whether the file starts with invisible characters

Using `repr()` instead of `print()` reveals characters that are normally invisible.

In [2]:
with open(SHOP_A, encoding="utf-8") as f:
    head = [next(f) for _ in range(3)]

for line in head:
    print(repr(line))

'VIN,RO_OPEN_DATE,RO_CLOSE_DATE,MILEAGE,ODOMETER_MEASURE,RO_INVOICE_NUMBER,SERVICE_DESCRIPTION,LABOR_DESCRIPTION,PART_NAME_DESCRIPTION,PART_QUANTITY,MAKE,MODEL,MODEL_YEAR,PLATE,PLATE_STATE,MANAGEMENT_SYSTEM,LOCATION_ID,LOCATION_NAME,ADDRESS,CITY,STATE,POSTAL_CODE,PHONE,URL\n'
'1FTFW1E50KFA12345,10/10/2019,10/10/2019,"21,000",KM,184200,"Lube oil and filter, 5W30 synthetic","LUBE OIL AND FILTER, 5W30 SYNTHETIC",OIL FILTER,1,FORD,F-150,2019,CJKT 421,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca\n'
'1FTFW1E50KFA12345,06/03/2020,06/04/2020,30733,KM,184203,Replace front brake pads and machine rotors,REPLACE FRONT BRAKE PADS AND MACHINE ROT,BRAKE PAD SET,2,FORD,F-150,2019,,,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca\n'


- [X] It starts directly with `b'1FTW...`, which means there is no BOM.
- [X] The line ending is `\r\n`.
- [X] Commas are used as the delimiter.
- [X] Quotes are used for quoted fields. They are applied when the text contains a comma, which prevents the parser from splitting that record.
- [X] All data appears to be ASCII, so UTF-8 should be valid. This will be verified in the next cell.

In [3]:
row = head[1]
print('splitting by commas:', len(row.split(',')), 'fields')
print('parsing as CSV:', len(next(csv.reader([row]))), 'fields')
print('fields in the header:', len(next(csv.reader([head[0]]))))

splitting by commas: 27 fields
parsing as CSV: 24 fields
fields in the header: 24


The first record shows that `MILEAGE` is `21,000`. Because it contains a comma, it is parsed as a `str`, and when the data is split by commas, the first row ends up with 25 fields instead of 24.

### 1.2 Data Reading

`shop_a_raw` is read once and **never modified again**. Any transformation is performed on a copy.

Without this rule, after three cells you can no longer compare the before and after, and the notebook can no longer be run from top to bottom.

In [4]:
shop_a_raw = pd.read_csv(SHOP_A)
shop_a = shop_a_raw.copy()

print(shop_a_raw.shape)
shop_a_raw.head(2)

(42, 24)


,VIN,RO_OPEN_DATE,RO_CLOSE_DATE,MILEAGE,ODOMETER_MEASURE,RO_INVOICE_NUMBER,SERVICE_DESCRIPTION,LABOR_DESCRIPTION,PART_NAME_DESCRIPTION,PART_QUANTITY,...,PLATE_STATE,MANAGEMENT_SYSTEM,LOCATION_ID,LOCATION_NAME,ADDRESS,CITY,STATE,POSTAL_CODE,PHONE,URL
0,1FTFW1E50KFA12345,10/10/2019,10/10/2019,"21,000",KM,184200,"Lube oil and filter, 5W30 synthetic","LUBE OIL AND FILTER, 5W30 SYNTHETIC",OIL FILTER,1.0,...,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca
1,1FTFW1E50KFA12345,06/03/2020,06/04/2020,30733,KM,184203,Replace front brake pads and machine rotors,REPLACE FRONT BRAKE PADS AND MACHINE ROT,BRAKE PAD SET,2.0,...,NaN,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca


### 1.3 Data Types

`read_csv` reads everything as text and then tries to convert each column to a number. If **a single value** in the column cannot be converted, it cancels the conversion for the entire column and leaves it as `object`. If there are `NaN` they are typed as `float`.

In [5]:
shop_a_raw.dtypes

VIN                       object
RO_OPEN_DATE              object
RO_CLOSE_DATE             object
MILEAGE                   object
ODOMETER_MEASURE          object
RO_INVOICE_NUMBER          int64
SERVICE_DESCRIPTION       object
LABOR_DESCRIPTION         object
PART_NAME_DESCRIPTION     object
PART_QUANTITY            float64
MAKE                      object
MODEL                     object
MODEL_YEAR               float64
PLATE                     object
PLATE_STATE               object
MANAGEMENT_SYSTEM         object
LOCATION_ID               object
LOCATION_NAME             object
ADDRESS                   object
CITY                      object
STATE                     object
POSTAL_CODE               object
PHONE                      int64
URL                       object
dtype: object

> For example, the fact that the data type of `MODEL_YEAR` is `float` indicates that there is at least one year recorded as a float or NaN.

### 1.4 What `object` dtype does not tell

`object` means pandas stopped making assumptions about that column. There could be one type or five, and the dtype looks the same. To know what is actually inside, you have to count the real data types.

In [6]:
print(shop_a_raw['MILEAGE'].map(type).value_counts())
print()
print('first values:', shop_a_raw['MILEAGE'].head(3).tolist())

MILEAGE
<class 'str'>      41
<class 'float'>     1
Name: count, dtype: int64

first values: ['21,000', '30733', '33255']


By counting the types of values in `MILEAGE`, we confirm that `read_csv` converted everything to `str`, with the only `float` being the `NaN`, as mentioned earlier. This shows that simply using `.dtype` does not tell us the actual type of a specific value, which is why we will use **Pydantic**.

### 1.5 Mapping dictionary

Here we declare the **source-to-target mapping (STTM)**: each source column names its target field, or `None` when it has no target.

It is written as data rather than logic for three reasons:

* Adding a column to the feed means adding one line here, not editing code.
* Mapping coverage can be counted and audited.
* Columns with `None` are exactly what the `E010` code counts.

The values come from `docs/DATA_DICTIONARY.md`. The dictionary does not invent anything; it implements it.

In [7]:
SHOP_A_MAP: dict[str, str | None] = {
    "VIN": "vin",
    "RO_OPEN_DATE": "event_date",
    "RO_CLOSE_DATE": None,
    "MILEAGE": "odometer_km",
    "ODOMETER_MEASURE": "odometer_source_unit",
    "RO_INVOICE_NUMBER": "source_record_id",
    "SERVICE_DESCRIPTION": "raw_description",
    "LABOR_DESCRIPTION": None,
    "PART_NAME_DESCRIPTION": None,
    "PART_QUANTITY": None,
    "MAKE": None,
    "MODEL": None,
    "MODEL_YEAR": None,
    "PLATE": None,
    "PLATE_STATE": None,
    "MANAGEMENT_SYSTEM": None,
    "LOCATION_ID": None,
    "LOCATION_NAME": "provider_name",
    "ADDRESS": None,
    "CITY": "provider_city",
    "STATE": "provider_province",
    "POSTAL_CODE": None,
    "PHONE": None,
    "URL": None,
}

#### Coverage audit

This check verifies that the columns in the file and the columns in the dictionary match.

If the file contains a column that the dictionary does not know about, or the dictionary mentions a column that no longer exists in the file, there is a problem with the mapping. Without this audit, the pipeline could keep running without showing any errors, and that column would simply disappear.

That is why this check helps detect incomplete or outdated mappings and protect data integrity.

This is literally *auditing mappings for data integrity*.

In [8]:
in_file = set(shop_a_raw.columns)
in_map = set(SHOP_A_MAP)

print('in the file and not in the dictionary:', in_file - in_map)
print('in the dictionary and not in the file:', in_map - in_file)
assert in_file == in_map, 'the dictionary and the file do not match'

mapped = [c for c, f in SHOP_A_MAP.items() if f is not None]
unmapped = [c for c, f in SHOP_A_MAP.items() if f is None]

print(f'\n{len(mapped)} mapped | {len(unmapped)} without target | coverage {len(mapped) / len(SHOP_A_MAP):.0%}')
print('\nwithout target (E010):', unmapped)

in the file and not in the dictionary: set()
in the dictionary and not in the file: set()

9 mapped | 15 without target | coverage 38%

without target (E010): ['RO_CLOSE_DATE', 'LABOR_DESCRIPTION', 'PART_NAME_DESCRIPTION', 'PART_QUANTITY', 'MAKE', 'MODEL', 'MODEL_YEAR', 'PLATE', 'PLATE_STATE', 'MANAGEMENT_SYSTEM', 'LOCATION_ID', 'ADDRESS', 'POSTAL_CODE', 'PHONE', 'URL']


#### Why the 15 columns without a target are not garbage

`MAKE`, `MODEL`, `MODEL_YEAR`, and `PLATE` could be used to match records for the same vehicle across feeds, and to verify that a VIN decodes to the correct vehicle. `POSTAL_CODE` and `ADDRESS` could be used to identify the shop when its name is written in different ways.

Reporting them as `E010` instead of ignoring them turns that gap into a **documented decision** rather than an oversight.

### 1.6 Profiling

**Profiling:** discover/explore -> What did I find in the data, and what does it imply?

**Validation:** decision. -> What conditions must a record meet to pass?

Example:

- VIN contains `NaN` -> **Profiling:** I discovered it.
- VIN with `NaN` is rejected -> **Validation:** I made a decision.
- MI and KM -> **Profiling:** I discovered two different units in one column.
- Converting MI to KM -> **Transformation:** I decided how to normalize them.

In [9]:
profile(shop_a_raw, mapped)

,column,nulls,unique,spare_spaces
0,VIN,0,26,0.0
1,RO_OPEN_DATE,0,39,0.0
2,MILEAGE,1,40,0.0
3,ODOMETER_MEASURE,0,2,0.0
4,RO_INVOICE_NUMBER,0,41,NaN
5,SERVICE_DESCRIPTION,1,33,8.0
6,LOCATION_NAME,0,3,0.0
7,CITY,0,1,0.0
8,STATE,0,1,0.0


#### Value inventory

Counting nulls is not enough for columns that determine code branches. So need to inspect the actual values.

In [10]:
for c in ['ODOMETER_MEASURE', 'LOCATION_NAME', 'CITY', 'STATE']:
    values = sorted(shop_a_raw[c].dropna().unique().tolist())
    print(f'{c:20} {len(values)} distinct -> {values}')

print('\nrows in miles:', int((shop_a_raw['ODOMETER_MEASURE'] == 'MI').sum()))

ODOMETER_MEASURE     2 distinct -> ['KM', 'MI']
LOCATION_NAME        3 distinct -> ['RIVERSIDE AUTO SERVICE', 'Riverside Auto Service', 'Riverside Auto Service Ltd']
CITY                 1 distinct -> ['London']
STATE                1 distinct -> ['ON']

rows in miles: 6


### 1.7 Duplicates within the feed

There are two levels of deduplication, and this is the cheap one: **an identical row within the same file**.

It can be detected without mapping anything. The expensive case—when the same event is represented differently or arrives through two feeds—requires the canonical schema and belongs in [`02_pipeline.ipynb`](02_pipeline.ipynb).

In [11]:
print('exact duplicate rows:', int(shop_a_raw.duplicated().sum()))
print('unique RO_INVOICE_NUMBER:', shop_a_raw['RO_INVOICE_NUMBER'].nunique(), 'of', len(shop_a_raw))

repeated = shop_a_raw[shop_a_raw['RO_INVOICE_NUMBER'].duplicated(keep=False)]
repeated[['RO_INVOICE_NUMBER', 'VIN', 'RO_OPEN_DATE', 'MILEAGE', 'SERVICE_DESCRIPTION', 'LABOR_DESCRIPTION']]

exact duplicate rows: 1
unique RO_INVOICE_NUMBER: 41 of 42


,RO_INVOICE_NUMBER,VIN,RO_OPEN_DATE,MILEAGE,SERVICE_DESCRIPTION,LABOR_DESCRIPTION
34,184302,1FA6P8TH2J5142608,01/29/2019,65220,Weld exhaust flex pipe,WELD EXHAUST FLEX PIPE
35,184302,1FA6P8TH2J5142608,01/29/2019,65220,Weld exhaust flex pipe,WELD EXHAUST FLEX PIPE


### 1.8 What the profiling forces us to write
Every rule below is justified by a finding from the previous cells, not by intuition.
* **`MILEAGE`**
  * Finding: 1 null, thousands separator, all values are strings.
  * Rule: remove the separator, use a nullable integer, convert when the unit is `MI`, and treat empty values and `0` as unknown.  
* **`ODOMETER_MEASURE`**
  * Finding: 2 values, no nulls.
  * Rule: convert to lowercase and use it to determine how `MILEAGE` should be converted.
* **`RO_INVOICE_NUMBER`**
  * Finding: 41 unique values out of 42 rows.
  * Rule: detect the exact duplicate and reject the second occurrence as `E009`.
* **`SERVICE_DESCRIPTION`**
  * Finding: 1 null, 8 values with extra spaces.
  * Rule: apply `strip`; an empty description becomes `E004`.
* **`LOCATION_NAME`**
  * Finding: 3 different spellings for the same shop.
  * Rule: standardize them to one form.
* **`RO_OPEN_DATE`**
  * Finding: no nulls, `MM/DD/YYYY` format.
  * Rule: parse using an explicit format; an invalid date becomes `E005`.
* **`VIN`**, **`CITY`**, **`STATE`**
  * Finding: clean.
  * Rule: copy them, with defensive `strip` and `upper`.

**Design decision from the profiling:**
`LOCATION_NAME` has 3 distinct values, while `LOCATION_ID` has only 1. The identifier is stable; the name is not. Therefore, the provider is identified by `LOCATION_ID`, while the name remains presentation text.

This decision goes into [`../docs/DESIGN_DECISIONS.md`](../docs/DESIGN_DECISIONS.md), together with the rejected alternative.


### 1.9 Summary of `shop_a`

* 42 rows, 24 columns.
* 9 columns have a target in the canonical schema; 15 are explicitly mapped to `None` and reported as `E010`.
* Issues found: 1 exact duplicate row, 1 empty odometer value, 1 empty description, 8 descriptions with extra spaces, and 3 spellings of the same shop.
* Feed trap: quoted values containing the separator.

**What remains for [`02_pipeline.ipynb`](02_pipeline.ipynb):** validate the VIN, parse the date, convert the odometer, and determine the outcome of each record.

## 2. A car dealership: `dealer_b` - XML
22 repair orders containing 27 jobs. The difference between those two numbers is the entire section.

### 1.1 Raw look

In [12]:
with open(DEALER_B, encoding="utf-8") as f:
    for _ in range(20):
        print(f.readline().rstrip())

<?xml version="1.0" encoding="UTF-8"?>
<ProcessRepairOrder xmlns="http://www.starstandard.org/STAR/5" versionID="5.3.4" systemEnvironmentCode="Production">
  <ApplicationArea>
    <Sender>
      <CreatorNameCode>DEALERTRACK</CreatorNameCode>
      <SenderNameCode>FCM-LONDON</SenderNameCode>
      <DealerNumberID>ON41127</DealerNumberID>
    </Sender>
    <CreationDateTime>2026-07-31T23:14:07-04:00</CreationDateTime>
    <BODID>a41f0c92-6b19-4d0e-9d2a-7c1f55a0b331</BODID>
  </ApplicationArea>
  <ProcessRepairOrderDataArea>
    <Process/>
    <RepairOrder>
      <RepairOrderHeader>
        <DocumentID>RO-100480</DocumentID>
        <SecondaryReferenceNumberString>RO-100480-A</SecondaryReferenceNumberString>
        <DealerParty>
          <PartyID>ON41127</PartyID>
          <OrganizationName>Forest City Motors</OrganizationName>


We can see that 
```xml
<ProcessRepairOrder xmlns="http://www.starstandard.org/STAR/5" versionID="5.3.4" systemEnvironmentCode="Production">
```
uses the full URL as the default namespace because it has no `star:` prefix, which idealy should be applied to the parent and all children:
```xml
<?xml version="1.0" encoding="UTF-8"?>
<star:ProcessRepairOrder xmlns:star="http://www.starstandard.org/STAR/5" versionID="5.3.4" systemEnvironmentCode="Production">
  <star:ApplicationArea>
    <star:Sender>
      <star:CreatorNameCode>DEALERTRACK</star:CreatorNameCode>
      <star:SenderNameCode>FCM-LONDON</star:SenderNameCode>
      <star:DealerNumberID>ON41127</star:DealerNumberID>
    </star:Sender>
      ...
```
For this reason we need to create a map of namespaces similar to:
```python
ns = {'star': 'http://www.starstandard.org/STAR/5'}
```
In order to python to understand the entire URL when I type `star`.

### 2.2 File hierarchy
We print the tree using these functions, showing each element name only once per level.

In [13]:
import xml.etree.ElementTree as ET

root = ET.parse(DEALER_B).getroot()


def tag(el) -> str:
    """Element name without its namespace prefix."""
    return el.tag.split("}")[-1]


def show_tree(el, depth: int = 0, max_depth: int = 6) -> None:
    if depth > max_depth:
        return
    print("   " * depth + tag(el))
    seen = set()
    for child in el:
        if tag(child) in seen:
            continue
        seen.add(tag(child))
        show_tree(child, depth + 1, max_depth)


show_tree(root)

ProcessRepairOrder
   ApplicationArea
      Sender
         CreatorNameCode
         SenderNameCode
         DealerNumberID
      CreationDateTime
      BODID
   ProcessRepairOrderDataArea
      Process
      RepairOrder
         RepairOrderHeader
            DocumentID
            SecondaryReferenceNumberString
            DealerParty
               PartyID
               OrganizationName
               AddressLine
               CityName
               StateOrProvinceCountrySubDivisionID
               PostalCode
            ServiceAdvisorParty
               PersonName
            LocationID
            DepartmentType
            RepairOrderOpenedDate
            RepairOrderCompletedDate
            RepairOrderStatus
         RepairOrderLineItem
            Vehicle
               VehicleID
               ModelYear
               MakeString
               ModelString
            LicenseNumberString
            InDistanceMeasure
            OutDistanceMeasure
            Job
         

`JobID` is contained within `Job`, which indicates that a repair order can contain multiple jobs. Each job must be separated or flattened into its own row in the canonical schema. This process is called **flattening**.

### 2.3 Will a direct search work?
The actual name of each element consists of the element name together with its namespace. Searching for `RepairOrder` alone, for example, looks for an element without a namespace, which does not exist in the document.

In [14]:
print('searching without namespace:', root.findall('RepairOrder'))

# namespace mapping
NS = {'star': root.tag.split('}')[0].lstrip('{')}
print('detected namespace:', NS)

print('searching with namespace:', len(root.findall('.//star:RepairOrder', NS)), 'repair orders')

searching without namespace: []
detected namespace: {'star': 'http://www.starstandard.org/STAR/5'}
searching with namespace: 22 repair orders


If we found this list to be empty, we would get no error, which would be worse because an error is easy to track down.

### 2.4 Number of events/jobs = Number of RepairOrders?
**No**, como mencionamos antes, una orden puede tener varios trabajos para el mismo vehiculo (mismo vin, vehicle, fecha

In [15]:
orders = root.findall('.//star:RepairOrder', NS)
jobs = root.findall('.//star:Job', NS)

print(len(orders), 'repair orders contain', len(jobs), 'jobs')
print()
print('jobs per repair order:')
print(pd.Series([len(o.findall('.//star:Job', NS)) for o in orders]).value_counts().sort_index())

22 repair orders contain 27 jobs

jobs per repair order:
1    17
2     5
Name: count, dtype: int64


This is important because if the feed were read as producing a single row per repair order, 5 jobs would be lost even in this simple example. That means jobs performed on the vehicle would be missing from its service history, potentially omitting information that could be critical for making certain decisions.

### 2.5 Flattening into a table: 1 row -> 1 job

The vehicle data, odometer, and dealership information live at the repair-order level. The job data lives at the job level. Each row combines the two levels.

`text()` returns `None` when the element does not exist or is empty, instead of raising an error. The check is `is None`, not `if not found`, because an XML element with no children evaluates to `False` even though it exists.


In [16]:
def text(el, path: str) -> str | None:
    """Stripped text at path, or None when the element is absent or empty."""
    found = el.find(path, NS)
    if found is None or found.text is None:
        return None
    return found.text.strip() or None


rows = []
for order in orders:
    distance = order.find(".//star:InDistanceMeasure", NS)
    header = {
        "DocumentID": text(order, ".//star:DocumentID"),
        "VehicleID": text(order, ".//star:VehicleID"),
        "RepairOrderOpenedDate": text(order, ".//star:RepairOrderOpenedDate"),
        "InDistanceMeasure": distance.text.strip() if distance is not None else None,
        "unitCode": distance.get("unitCode") if distance is not None else None,
        "OrganizationName": text(order, ".//star:OrganizationName"),
        "CityName": text(order, ".//star:CityName"),
        "StateOrProvinceCountrySubDivisionID": text(
            order, ".//star:StateOrProvinceCountrySubDivisionID"
        ),
    }
    for job in order.findall(".//star:Job", NS):
        rows.append(
            {
                **header,
                "JobID": text(job, "star:JobID"),
                "CustomerConcernDescription": text(job, "star:CustomerConcernDescription"),
                "CorrectionDescription": text(job, "star:CorrectionDescription"),
            }
        )

dealer_b_raw = pd.DataFrame(rows)
print(dealer_b_raw.shape)
dealer_b_raw.head(3)

(27, 11)


,DocumentID,VehicleID,RepairOrderOpenedDate,InDistanceMeasure,unitCode,OrganizationName,CityName,StateOrProvinceCountrySubDivisionID,JobID,CustomerConcernDescription,CorrectionDescription
0,RO-100480,1FTFW1E50KFA12345,2021-03-14T08:20:00-04:00,41199,KMT,Forest City Motors,London,ON,1,Grinding noise when braking,"R&R rear brake pads, resurface rotors"
1,RO-100487,2T1BURHE4JC021345,2021-05-02T08:21:00-04:00,55542,KMT,FOREST CITY MOTORS,London,ON,1,Vehicle pulls to the right,Replace front struts and mounts
2,RO-100494,1HGCV1F30LA100324,2022-08-03T08:22:00-04:00,29120,SMI,Forest City Motors Ltd,London,ON,1,"Engine light on, rough running",Replace spark plugs and coil on cyl 3


This specific flattening has one constraint:

**1 repair order -> 1 vehicle -> 1 odometer reading**

If an order with two vehicles ever arrives, `.//star:VehicleID` returns the first one and the second is silently lost. Therefore, we check for this condition and raise a warning.

In [17]:
assert len(dealer_b_raw) == len(jobs), 'jobs were lost during flattening'

for name in ['VehicleID', 'InDistanceMeasure', 'RepairOrderLineItem']:
    per_order = {len(o.findall(f'.//star:{name}', NS)) for o in orders}
    print(f'{name:22} per order: {per_order}')

VehicleID              per order: {1}
InDistanceMeasure      per order: {1}
RepairOrderLineItem    per order: {1}


### 2.6 Mapping dictionary
Here, `E010` works differently than it does in the CSV. In a CSV, all columns arrive in the table, so any unused columns are visible. In an XML, only the elements we extract make it into the table, so anything that is not mapped is invisible **unless we explicitly manually declared**.

In [18]:
DEALER_B_MAP: dict[str, str | None] = {
    'DocumentID': 'source_record_id',
    'JobID': 'source_record_id',
    'VehicleID': 'vin',
    'RepairOrderOpenedDate': 'event_date',
    'InDistanceMeasure': 'odometer_km',
    'unitCode': 'odometer_source_unit',
    'CorrectionDescription': 'raw_description',
    'CustomerConcernDescription': 'raw_description',
    'OrganizationName': 'provider_name',
    'CityName': 'provider_city',
    'StateOrProvinceCountrySubDivisionID': 'provider_province',
}

DEALER_B_UNMAPPED = [
    'SecondaryReferenceNumberString',
    'ServiceAdvisorParty',
    'LocationID',
    'DepartmentType',
    'RepairOrderStatus',
    'RepairOrderCompletedDate',
    'LicenseNumberString',
    'OutDistanceMeasure',
    'ServiceLaborOperationCode',
    'LaborActualHoursNumeric',
    'ApplicationArea',
]

assert set(DEALER_B_MAP) == set(dealer_b_raw.columns), 'dictionary and table do not match'

print('> Declared unmapped elements and their actual presence in the file:')
for name in DEALER_B_UNMAPPED:
    print(f'  {name:32} {len(root.findall(".//star:" + name, NS)):>3}')

> Declared unmapped elements and their actual presence in the file:
  SecondaryReferenceNumberString    22
  ServiceAdvisorParty               22
  LocationID                        22
  DepartmentType                    22
  RepairOrderStatus                 22
  RepairOrderCompletedDate          21
  LicenseNumberString               22
  OutDistanceMeasure                22
  ServiceLaborOperationCode         27
  LaborActualHoursNumeric           27
  ApplicationArea                    1


> This list verifies that every element declared as unmapped actually exists in the file. If any of them appears with zero occurrences, the list is outdated and the `E010` report would be inaccurate.

### 2.7 Profiling

In [19]:
profile(dealer_b_raw, list(dealer_b_raw.columns))

,column,nulls,unique,spare_spaces
0,DocumentID,0,22,0
1,VehicleID,0,22,0
2,RepairOrderOpenedDate,0,22,0
3,InDistanceMeasure,0,22,0
4,unitCode,0,2,0
5,OrganizationName,0,3,0
6,CityName,0,1,0
7,StateOrProvinceCountrySubDivisionID,0,1,0
8,JobID,0,2,0
9,CustomerConcernDescription,1,13,0


In [20]:
for c in ['unitCode', 'OrganizationName', 'CityName', 'StateOrProvinceCountrySubDivisionID']:
    values = sorted(dealer_b_raw[c].dropna().unique().tolist())
    print(f'{c:38} {len(values)} distinct -> {values}')

unitCode                               2 distinct -> ['KMT', 'SMI']
OrganizationName                       3 distinct -> ['FOREST CITY MOTORS', 'Forest City Motors', 'Forest City Motors Ltd']
CityName                               1 distinct -> ['London']
StateOrProvinceCountrySubDivisionID    1 distinct -> ['ON']


In [21]:
correction = dealer_b_raw['CorrectionDescription']
concern = dealer_b_raw['CustomerConcernDescription']

print('both present          :', int((correction.notna() & concern.notna()).sum()))
print('correction only       :', int((correction.notna() & concern.isna()).sum()))
print('customer concern only :', int((correction.isna() & concern.notna()).sum()))
print('neither               :', int((correction.isna() & concern.isna()).sum()))

dealer_b_raw.loc[correction.notna() & concern.notna(), ['CustomerConcernDescription', 'CorrectionDescription']].head(3)

both present          : 26
correction only       : 0
customer concern only : 0
neither               : 1


,CustomerConcernDescription,CorrectionDescription
0,Grinding noise when braking,"R&R rear brake pads, resurface rotors"
1,Vehicle pulls to the right,Replace front struts and mounts
2,"Engine light on, rough running",Replace spark plugs and coil on cyl 3


Here it is important to determine which rule to create. `concern`, `correction`, either one, or both may be empty. We need to decide which one should be used as the job description in the canonical schema.

Which one **DESCRIBES** the work better?

If we use `concern`, we would produce a history of customer complaints. Therefore, we will use `correction` so that instead of something like *'brake noise'*, the description gives us something like *'replaced rear brake pads'*, this one is more valuable for the VHR. However, we still preserve `concern`.

If neither is present, the record receives `E004`. If only `concern` is present, we use it.

Hierarchy: `correction` -> `concern` -> `E004`

### 2.9 Rules and summary of `dealer_b`

* **`DocumentID` + `JobID`**
  * Finding: neither is unique on its own.
  * Rule: combine them with a hyphen, e.g. `RO-100480-J1`
* **`VehicleID`**
  * Finding: clean.
  * Rule: apply defensive `strip` and `upper`.
* **`RepairOrderOpenedDate`**
  * Finding: ISO timestamp with an offset.
  * Rule: parse with the offset and keep only the date.
* **`InDistanceMeasure`**
  * Finding: 2 units identified through `unitCode`.
  * Rule: convert `SMI` to `mi` and convert it to `km`; convert `KMT` directly to `km`.
* **`CorrectionDescription`**
  * Finding: missing in some jobs.
  * Rule: use the correction first, the customer concern as a fallback, and assign `E004` if neither is present.
* **`OrganizationName`**
  * Finding: 3 spellings of the same dealership.
  * Rule: standardize them to one form.
* 22 repair orders contain 27 jobs, and each job becomes an event.
* Feed trap: the namespace causes a direct search to return an empty result instead of raising an error.
* The event identifier does not exist in the file; it is constructed by combining two fields.

## 3. A fleet agreggator: `fleet_c` - JSON
28 records inside an envelope. The most modern of the three feeds, and the one with the worst data types.

### 3.1 Raw Look
The top level is not the table. It is an envelope with two parts: response metadata and the records array.

In [33]:
import json

with open(FLEET_C, encoding='utf-8') as f:
    fleet_c_doc = json.load(f)

for key, value in fleet_c_doc.items():
    size = len(value) if isinstance(value, (list, dict)) else value
    print(f'{key:10} {type(value).__name__:6} {size}')

print(f"\nmeta values: {fleet_c_doc['meta']}")

meta       dict   5
records    list   28

meta values: {'api_version': 'v2', 'generated_at': '2026-07-31T04:00:11Z', 'page': 1, 'page_size': 100, 'total_records': 28}


In [34]:
try:
    pd.read_json(FLEET_C)
except ValueError as error:
    print("ValueError:", error)

ValueError: Mixing dicts with non-Series may lead to ambiguous ordering.


We cannot read the JSON directly using the pandas function because it would try to use `meta` and `records` as columns, since they are at the top level. We need to access `records` first.

```json
{'meta': {
    'api_version': 'v2',
    'generated_at': '2026-07-31T04:00:11Z',
    'page': 1,
    'page_size': 100,
    'total_records': 28},
 'records': [
    {
        'work_order_id': 'WO-2026-4100',
        'vin': '2T1BURHE4JC021345',
        'plate': 'BXRP 887',
        'service_date': '2018-11-23T00:00:00Z',
        'odometer': {
            'value': 23211,
            'unit': 'km'},
        'description': 'Mount and balance 4 winter tires',
        'vendor': {
            'id': 'V-3301',
            'name': 'NorthStar Fleet Services',
            'city': 'Windsor',
            'region': 'ON'},
        'invoice_total_cad': 120.0,
        'cost_centre': 'CC-200'
    },
    ...
```

### 3.2 From Nested Records to a Table
We should use `json_normalize` to flatten the nested objects once we access the `records`. No matter how many levels there are, it will flatten them using this naming convention: `grandparent.parent.child.grandchild...`

In [45]:
fleet_c_raw = pd.json_normalize(fleet_c_doc["records"]) # sep='.' per default
fleet_c_raw.head(2)

,work_order_id,vin,plate,service_date,description,invoice_total_cad,cost_centre,odometer.value,odometer.unit,vendor.id,vendor.name,vendor.city,vendor.region
0,WO-2026-4100,2T1BURHE4JC021345,BXRP 887,2018-11-23T00:00:00Z,Mount and balance 4 winter tires,120.0,CC-200,23211,km,V-3301,NorthStar Fleet Services,Windsor,ON
1,WO-2026-4105,2T1BURHE4JC021345,BXRP 887,2020-05-23T00:00:00Z,Four wheel alignment,157.4,CC-201,"44,032",km,V-3318,Dominion Vehicle Care,Kitchener,ON


In [56]:
expected = fleet_c_doc['meta']['total_records'] # quantity declared by company as a metadata
assert expected == len(fleet_c_doc['records']), 'length does not match'

### 3.4 Mapping dictionary

In [60]:
FLEET_C_MAP: dict[str, str | None] = {
    'work_order_id': 'source_record_id',
    'vin': 'vin',
    'plate': None,
    'service_date': 'event_date',
    'description': 'raw_description',
    'invoice_total_cad': None,
    'cost_centre': None,
    'odometer.value': 'odometer_km',
    'odometer.unit': 'odometer_source_unit',
    'vendor.id': None,
    'vendor.name': 'provider_name',
    'vendor.city': 'provider_city',
    'vendor.region': 'provider_province',
}

assert set(FLEET_C_MAP) == set(fleet_c_raw.columns), 'mapping dictionary and table do not match'

mapped_c = [c for c, f in FLEET_C_MAP.items() if f is not None]
unmapped_c = [c for c, f in FLEET_C_MAP.items() if f is None]

print(f'{len(mapped_c)} mapped | {len(unmapped_c)} without destination | coverage {len(mapped_c) / len(FLEET_C_MAP):.0%}')
print('without destination (E010):', unmapped_c, 'plus the entire meta block')

9 mapped | 4 without destination | coverage 69%
without destination (E010): ['plate', 'invoice_total_cad', 'cost_centre', 'vendor.id'] plus the entire meta block


### 3.5 Feed trap: type change between records

In [62]:
print(fleet_c_raw['odometer.value'].map(type).value_counts())
print()
print('column dtype:', fleet_c_raw['odometer.value'].dtype)
print()
for i in [0, 1, 6]:
    value = fleet_c_raw['odometer.value'].iloc[i]
    print(f"  {fleet_c_raw['work_order_id'].iloc[i]:14} {str(value):10} {type(value).__name__}")

odometer.value
<class 'int'>         23
<class 'str'>          3
<class 'NoneType'>     2
Name: count, dtype: int64

column dtype: object

  WO-2026-4100   23211      int
  WO-2026-4105   44,032     str
  WO-2026-4130   None       NoneType


### 3.6 Profiling

In [72]:
profile(fleet_c_raw, mapped_c)

,column,nulls,unique,spare_spaces
0,work_order_id,0,28,0
1,vin,0,18,0
2,service_date,0,28,0
3,description,1,19,0
4,odometer.value,2,26,0
5,odometer.unit,0,2,0
6,vendor.name,0,3,0
7,vendor.city,0,3,0
8,vendor.region,0,1,0


In [74]:
for c in ['odometer.unit', 'vendor.name', 'vendor.city', 'vendor.region']:
    values = sorted(fleet_c_raw[c].dropna().unique().tolist())
    print(f'{c:16} {len(values)} distinct -> {values}')

print()
print('VIN in lowercase:', fleet_c_raw.loc[fleet_c_raw['vin'] != fleet_c_raw['vin'].str.upper(), 'vin'].tolist())
print('null description:', fleet_c_raw.loc[fleet_c_raw['description'].isna(), 'work_order_id'].tolist())

odometer.unit    2 distinct -> ['km', 'mi']
vendor.name      3 distinct -> ['Dominion Vehicle Care', 'Great Lakes Fleet Maintenance', 'NorthStar Fleet Services']
vendor.city      3 distinct -> ['Kitchener', 'Sarnia', 'Windsor']
vendor.region    1 distinct -> ['ON']

VIN in lowercase: ['3vwc57bu8km052318']
null description: ['WO-2026-4180']


### 3.7 Rules and summary of `fleet_c`

* **`odometer.value`**
  * Finding: 3 types: number, string with separator, and null.
  * Rule: normalize to text, remove the separator, and convert to a nullable integer.
* **`odometer.unit`**
  * Finding: 2 units.
  * Rule: convert when the unit is `mi` and preserve the original unit.
* **`vin`**
  * Finding: 1 value is lowercase.
  * Rule: apply `upper` before validation; the data specifically requires this rule.
* **`service_date`**
  * Finding: ISO timestamp in UTC.
  * Rule: parse it, keep only the date, and assign `E005` for an invalid date.
* **`description`**
  * Finding: 1 null.
  * Rule: apply `strip`; an empty description becomes `E004`.
* **`vendor.name`**
  * Finding: 3 distinct vendors, with no spelling variations.
  * Rule: copy as-is; no standardization is needed.
* 28 records are contained in one envelope, verified against `meta.total_records`.
* 9 fields are mapped, 4 have no target, plus the entire `meta` block.
* Feed trap: data types change between records because JSON allows each value to have its own type.
* Unlike the other two feeds, `vendor.name` is already clean, so the provider-standardization rule **does not apply here**. A rule that is copied without measuring the data is a rule that does not belong.

## 4. What this notebook has ready

| Feed       |                         Records |    Mapped |             Unmapped | Format trap                           |
| ---------- | ------------------------------: | --------: | -------------------: | ------------------------------------- |
| `shop_a`   |                              42 | 9 columns |           15 columns | quotes around the separator           |
| `dealer_b` | 27 jobs across 22 repair orders | 11 fields |          11 elements | namespace and multiple jobs per order |
| `fleet_c`  |                              28 |  9 fields | 4 fields plus `meta` | data type changes between records     |

97 records in total. This is the number we will use to measure the pipeline.

> Each format has a different trap, and none of them raises an error. The CSV shifts columns, the XML returns an empty result, and the JSON changes data types. All three can produce a result that looks correct.
